<a href="https://colab.research.google.com/github/nyp-sit/iti121-2025s2/blob/main/finetune_embedding_llamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Embeddings


Finetuning embedding models often heavily improves the performance of the model on your use case, because each task requires a different notion of similarity. For example, given news articles:
- “Apple launches the new iPad”
- “NVIDIA is gearing up for the next GPU generation”
Then the following use cases, we may have different notions of similarity:
- a model for classification of news articles as Economy, Sports, Technology, Politics, etc., should produce similar embeddings for these texts.
- a model for semantic textual similarity should produce dissimilar embeddings for these texts, as they have different meanings.
- a model for semantic search would not need a notion for similarity between two documents, as it should only compare queries and documents.

In this notebook, we show users how to easily finetune their own embedding models using LLamaIndex.

We go through three main sections:
1. Preparing the data (our `generate_qa_embedding_pairs` function makes this easy)
2. Finetuning the model (using our `SentenceTransformersFinetuneEngine`)
3. Evaluating the model on a validation knowledge corpus

## Generate Corpus

First, we create the corpus of text chunks by leveraging LlamaIndex to load some financial PDFs, and parsing/chunking into plain text chunks.

In [ ]:
# %pip install datasets
%pip -q install llama-index-llms-openai
%pip -q install llama-index-embeddings-openai
%pip -q install llama-index-embeddings-azure-openai
%pip -q install llama-index-llms-azure-openai
%pip -q install llama-index-finetuning
%pip -q install llama-index-readers-file
%pip -q install llama-index-embeddings-huggingface
%pip -q install openai

# also need to import the generate_embedding_qa_pairs_batch() function
!wget -q https://raw.githubusercontent.com/nyp-sit/iti121-2025s2/refs/heads/main/L11/batch_generate.py

In [ ]:
import json
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode

Now let's us download two PDFs (annual report of Uber and Lyft filed with US Securities and Exchange Commission), and use them to generate some queries and answer pairs.

In [ ]:
!mkdir -p 'data/10k/'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/lyft_2021.pdf' -O 'data/10k/lyft_2021.pdf'

In [ ]:
TRAIN_FILES = ["./data/10k/lyft_2021.pdf"]
VAL_FILES = ["./data/10k/uber_2021.pdf"]

TRAIN_CORPUS_FPATH = "./data/train_corpus.json"
VAL_CORPUS_FPATH = "./data/val_corpus.json"

We will use [SimpleDirectoryReader](https://developers.llamaindex.ai/python/framework/module_guides/loading/simpledirectoryreader/) to read the PDF files and parse the files into text and use [SentenceSplitter](https://developers.llamaindex.ai/python/framework-api-reference/node_parsers/sentence_splitter/#llama_index.core.node_parser.SentenceSplitter) to split the text into text chunks.  LllamaIndex store the text chunks, together wiht the metadata, into data structure called [Nodes](https://developers.llamaindex.ai/python/framework/module_guides/loading/documents_and_nodes/).

In [ ]:
def load_corpus(files, verbose=False):
    if verbose:
        print(f"Loading files {files}")

    reader = SimpleDirectoryReader(input_files=files)
    docs = reader.load_data()
    if verbose:
        print(f"Loaded {len(docs)} docs")

    parser = SentenceSplitter()
    nodes = parser.get_nodes_from_documents(docs, show_progress=verbose)

    if verbose:
        print(f"Parsed {len(nodes)} nodes")

    return nodes

We do a very naive train/val split by having the Lyft corpus as the train dataset, and the Uber corpus as the val dataset.

In [ ]:
train_nodes = load_corpus(TRAIN_FILES, verbose=True)
val_nodes = load_corpus(VAL_FILES, verbose=True)

### Generate synthetic queries

Now, we use an LLM (e.g. gpt-4.1-nano) to generate questions using each text chunk in the corpus as context.

Each pair of (generated question, text chunk used as context) becomes a datapoint in the finetuning dataset (either for training or evaluation).

LlamaIndex provides a convenient method called `generate_qa_embedding_pairs()` to generate question and answer pairs. This method will invoke LLM for each text chunk and will be quite slow and costly. We have provided an implementation `generate_qa_embedding_pairs_batch()` that make use of the batch API calls of OpenAI to do batch processing, which is much more cost-efficient and potentially faster too.

In [ ]:
from batch_generate import generate_qa_embedding_pairs_batch
# from llama_index.finetuning.embeddings.common import generate_qa_embedding_pairs
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

In [ ]:
from openai import OpenAI

# generate_qa_embedding_pairs_batch requires an OpenAI client instead of the abstract LLM of LlamaIndex
endpoint = "https://nypopenai2.cognitiveservices.azure.com/openai/v1/"
model_name = "gpt-4.1-nano"
deployment_name = "gpt-4.1-nano"
api_key = "<<apikey>>"
openAIclient = OpenAI(
    base_url=f"{endpoint}",
    api_key=api_key
)

The `generate_qa_embedding_pairs_batch()` is implemented as non-blocking async function.  We will call it twice, one to generate train_dataset, and one to generate validation_dataset.  We will then wait for both to complete, using `await/gather`. This way, we can allow both jobs to run concurrently, which will save quite a bit of waiting time.

In [ ]:
import asyncio

results = await asyncio.gather(
    generate_qa_embedding_pairs_batch(
        nodes=train_nodes,
        openai_client=openAIclient,
        model="gpt-4.1-nano",
        num_questions_per_chunk=2,
        output_path="train_dataset.json"),
    generate_qa_embedding_pairs_batch(
        nodes=val_nodes,
        openai_client=openAIclient,
        model="gpt-4.1-nano",
        num_questions_per_chunk=2,
        output_path="val_dataset.json")
)

In [ ]:
# unpack the results into train/val dataset
train_dataset, val_dataset = results

In [ ]:
# [Optional] Load
train_dataset = EmbeddingQAFinetuneDataset.from_json("train_dataset.json")
val_dataset = EmbeddingQAFinetuneDataset.from_json("val_dataset.json")

## Run Embedding Finetuning

In this exercise, we will finetune an open source embedding model called `BAAI/bge-small-en`.  LlamaIndex provides the SentenceTransformersFinetuneEngine for finetuning purpose. Internally, it format the provided train dataset into the following format expected by SentenceTransformer for finetuning:
`(anchor, positive) pairs`
and uses MultipleNegativesRankingLoss as loss function to train.
The data format that is required for different chosen loss function is listed in the [loss table](https://sbert.net/docs/sentence_transformer/loss_overview.html) in SentenceTransformer website.

In [ ]:
from llama_index.finetuning import SentenceTransformersFinetuneEngine

In [ ]:
finetune_engine = SentenceTransformersFinetuneEngine(
    train_dataset,
    model_id="BAAI/bge-small-en",
    model_output_path="test_model",
    val_dataset=val_dataset,
)

You will need wandb API key for the finetuning process.  Please goto your wandb account and copy the api key, when prompted during the training process.

In [ ]:
import os

os.environ["WANDB_API_KEY"]="<<wandb_api_key"

In [ ]:
finetune_engine.finetune()

In [ ]:
embed_model = finetune_engine.get_finetuned_model()

## Evaluate Finetuned Model

In this section, we evaluate 3 different embedding models:
1. proprietary OpenAI embedding,
2. open source `BAAI/bge-small-en`, and
3. our finetuned embedding model.

We consider 2 evaluation approaches:
1. a simple custom **hit rate** metric
2. using `InformationRetrievalEvaluator` from sentence_transformers

We show that finetuning on synthetic (LLM-generated) dataset significantly improve upon an opensource embedding model.

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import TextNode
from tqdm.notebook import tqdm
import pandas as pd

### Define eval function

**Option 1**: We use a simple **hit rate** metric for evaluation:
* for each (query, relevant_doc) pair,
* we retrieve top-k documents with the query,  and
* it's a **hit** if the results contain the relevant_doc.

This approach is very simple and intuitive, and we can apply it to both the proprietary OpenAI embedding as well as our open source and fine-tuned embedding models.

In [ ]:
def evaluate(
    dataset,
    embed_model,
    top_k=5,
    verbose=False,
):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    nodes = [TextNode(id_=id_, text=text) for id_, text in corpus.items()]
    index = VectorStoreIndex(
        nodes, embed_model=embed_model, show_progress=True
    )
    retriever = index.as_retriever(similarity_top_k=top_k)

    eval_results = []
    for query_id, query in tqdm(queries.items()):
        retrieved_nodes = retriever.retrieve(query)
        retrieved_ids = [node.node.node_id for node in retrieved_nodes]
        expected_id = relevant_docs[query_id][0]
        is_hit = expected_id in retrieved_ids  # assume 1 relevant doc

        eval_result = {
            "is_hit": is_hit,
            "retrieved": retrieved_ids,
            "expected": expected_id,
            "query": query_id,
        }
        eval_results.append(eval_result)
    return eval_results

**Option 2**: We use the `InformationRetrievalEvaluator` from sentence_transformers.

This provides a more comprehensive suite of metrics, but we can only run it against the sentencetransformers compatible models (open source and our finetuned model, *not* the OpenAI embedding model).

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer
from pathlib import Path


def evaluate_st(
    dataset,
    model_id,
    name,
):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    evaluator = InformationRetrievalEvaluator(
        queries, corpus, relevant_docs, name=name
    )
    model = SentenceTransformer(model_id)
    output_path = "results/"
    Path(output_path).mkdir(exist_ok=True, parents=True)
    return evaluator(model, output_path=output_path)

### Run Evals

#### OpenAI

Note: this might take a few minutes to run since we have to embed the corpus and queries

In [ ]:
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding

openai_embed = AzureOpenAIEmbedding(
        model="text-embedding-3-small",
        api_key="<<apikey>>",
        azure_endpoint="https://nypopenai2.cognitiveservices.azure.com/",
        deployment_name="text-embedding-3-small",
        api_version="2023-05-15"
    )

In [ ]:
openai_embed
openai_embed_val_results = evaluate(val_dataset, openai_embed)

In [ ]:
df_openai_embed = pd.DataFrame(openai_embed_val_results)

In [ ]:
hit_rate_openai_embed = df_openai_embed["is_hit"].mean()
hit_rate_openai_embed

### BAAI/bge-small-en

In [ ]:
bge = "local:BAAI/bge-small-en"
bge_val_results = evaluate(val_dataset, bge)

In [ ]:
df_bge = pd.DataFrame(bge_val_results)

In [ ]:
hit_rate_bge = df_bge["is_hit"].mean()
hit_rate_bge

In [ ]:
evaluate_st(val_dataset, "BAAI/bge-small-en", name="bge")

### Finetuned

In [ ]:
finetuned = "local:test_model"
val_results_finetuned = evaluate(val_dataset, finetuned)

In [ ]:
df_finetuned = pd.DataFrame(val_results_finetuned)

In [ ]:
hit_rate_finetuned = df_finetuned["is_hit"].mean()
hit_rate_finetuned

In [ ]:
evaluate_st(val_dataset, "test_model", name="finetuned")

### Summary of Results

#### Hit rate

In [ ]:
df_openai_embed["model"] = "openai_embed"
df_bge["model"] = "bge"
df_finetuned["model"] = "fine_tuned"

We can see that fine-tuning our small open-source embedding model drastically improve its retrieval quality (even approaching the quality of the proprietary OpenAI embedding)!

In [ ]:
df_all = pd.concat([df_openai_embed, df_bge, df_finetuned])
df_all.groupby("model").mean("is_hit")

#### InformationRetrievalEvaluator

In [ ]:
df_st_bge = pd.read_csv(
    "results/Information-Retrieval_evaluation_bge_results.csv"
)
df_st_finetuned = pd.read_csv(
    "results/Information-Retrieval_evaluation_finetuned_results.csv"
)

We can see that embedding finetuning improves metrics consistently across the suite of eval metrics

In [ ]:
df_st_bge["model"] = "bge"
df_st_finetuned["model"] = "fine_tuned"
df_st_all = pd.concat([df_st_bge, df_st_finetuned])
df_st_all = df_st_all.set_index("model")
df_st_all

## Additional notes on the evaluation metrics

---

### **High-level summary**

Your finetuned embedding model is performing **reasonably well**:

* **Top-1 accuracy / recall @1 = 56%** → For more than half of the queries, the correct document is the first result.
* **Recall improves strongly at @3, @5, @10**, ending at **~87%**.
* **NDCG (rank quality) and MRR (ranking usefulness)** are both good (~0.7 and ~0.67).
* **Precision drops** as k increases — which is normal.

Overall, the model retrieves the right item most of the time, but may not always rank it at the very top.

---

### **Metric-by-metric explanation**

#### **1. Accuracy @k (same as Recall @k)**

This measures:

> *For each query, is the correct document found in the top-k retrieved results?*

| Metric          | Meaning                  | Your Score         |
| --------------- | ------------------------ | ------------------ |
| **accuracy@1**  | correct doc is at rank 1 | **0.5646** → 56.5% |
| **accuracy@3**  | in top 3                 | **0.7557** → 75.6% |
| **accuracy@5**  | in top 5                 | **0.8114** → 81.1% |
| **accuracy@10** | in top 10                | **0.8696** → 87%   |

##### Interpretation

This is quite decent. The model gets the correct answer somewhere in the top-10 nearly **9 out of 10** times.

Accuracy@1 = 56% means your ranking still has room for improvement.

---

#### **2. Precision @k**

This measures how many of the top-k results **are correct**, averaged across queries.

Since normally there is **only 1 relevant document per query**, precision@k will always be:

[
\text{precision@k} = \frac{\text{accuracy@k}}{k}
]

Which matches your numbers:

* **precision@1 = 0.5646** (same as accuracy@1)
* **precision@3 = 0.2519** ≈ 0.7557 / 3
* **precision@5 = 0.1623**
* **precision@10 = 0.0869**

##### Interpretation

Low precision is **normal** and expected when there's only **one relevant document**.

Precision is not very informative in single-relevance retrieval tasks.

---

#### **3. Recall @k**

Same as accuracy@k for single-relevant IR datasets.
(Already explained above.)

---

#### **4. NDCG@10 = 0.7178**

Normalized Discounted Cumulative Gain.

This measures **how well ranked** the relevant results are (penalizes correct docs that appear too low).

* Score **0.72** is good.
* If perfect, NDCG would be 1.0.

##### Interpretation

Your model does a solid job ranking the relevant doc near the top, not just retrieving it.

---

#### **5. MRR@10 = 0.6690**

Mean Reciprocal Rank.

Measures how early the first relevant document appears.

Example:

* If correct result is rank 1 → score = 1
* rank 2 → 1/2 = 0.5
* rank 5 → 1/5 = 0.2

Your average = **0.669** → ~1.49 rank-equivalent.

##### Interpretation

On average, the model ranks the correct document **between rank 1 and 2**.
This is strong performance.

---

#### **6. MAP@100 = 0.6733**

Mean Average Precision.

For single-relevant datasets, MAP acts similarly to MRR, but:

* Considers precision at all retrieved positions.
* Rewards models that place relevant doc as high as possible.

Your score **0.673** is consistent with MRR (0.669), which is expected.

##### Interpretation

Good retrieval ranking consistency across queries.

---

### **Overall Interpretation**

#### Strengths

✔ Correct document found in top-10: **87%**
✔ Top-1 rank is achieved in **56%** of queries
✔ Ranking metrics (NDCG/MRR/MAP ≈ 0.67–0.72) are strong
✔ Finetuning is successful relative to typical baselines

#### Weaknesses

✘ Not perfect at ranking — ~44% of queries fail to put the correct doc at rank 1
✘ Precision@k is low (but expected)

---

#### Next steps if you want to improve performance

Depending on your dataset:

### If you want better rank-1 accuracy:

* Use **hard negative mining** (especially for SBERT).
* Fine-tune with **Multiple Negatives Ranking Loss**.
* Ensure query–document pairs are clean and unambiguous.

### If queries differ from documents:

Use separate encoders:

* **Bi-encoder with dual encoders** (query encoder + doc encoder)
* Or apply QE (Query Embedding) instructions like prefix `"query: "` and `"doc: "`.
